# BioJev Sprint 2 — BioQwen domain adaptation

Execute the cells in order. Training/evaluation cells have explicit boolean guards because they can download a ~9 GB model and run long experiments.


In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Repository:", ROOT)
print("Python:", sys.executable)


## 1. Check PyTorch / CUDA


In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


## 2. Prepare/verify Sprint-1 datasets

The downloader now ends with a table summarizing all prepared datasets and splits.


In [ ]:
subprocess.run([sys.executable, "scripts/download_datasets.py", "--all"], check=True)


## 3. Inspect the streaming PubMed/PMC corpus


In [ ]:
subprocess.run([
    sys.executable, "scripts/inspect_corpus.py",
    "--config", "configs/dapt/smoke.yaml",
    "--documents", "5", "--show-text"
], check=True)


## 4. Smoke DAPT

Set `RUN_SMOKE_TRAINING=True` when ready. This uses the real Qwen3.5-4B-Base checkpoint with a 50K-token budget.


In [ ]:
RUN_SMOKE_TRAINING = False
if RUN_SMOKE_TRAINING:
    subprocess.run([
        sys.executable, "scripts/train_dapt.py",
        "--config", "configs/dapt/smoke.yaml"
    ], check=True)


## 5. 100M-token BioQwen run


In [ ]:
RUN_100M_TRAINING = False
if RUN_100M_TRAINING:
    subprocess.run([
        sys.executable, "scripts/train_dapt.py",
        "--config", "configs/dapt/pubmed_pmc_100m.yaml"
    ], check=True)


## 6. Resume after interruption


In [ ]:
RESUME_100M = False
if RESUME_100M:
    subprocess.run([
        sys.executable, "scripts/train_dapt.py",
        "--config", "configs/dapt/pubmed_pmc_100m.yaml",
        "--resume"
    ], check=True)


## 7. Held-out biomedical perplexity — Qwen3.5-4B-Base


In [ ]:
RUN_PERPLEXITY = False
if RUN_PERPLEXITY:
    subprocess.run([
        sys.executable, "scripts/evaluate_perplexity.py",
        "--model", "Qwen/Qwen3.5-4B-Base",
        "--config", "configs/dapt/pubmed_pmc_100m.yaml",
        "--partition", "validation",
        "--tokens", "1000000",
        "--output", "results/perplexity/qwen35_4b_base.json"
    ], check=True)


## 8. Held-out biomedical perplexity — BioQwen


In [ ]:
if RUN_PERPLEXITY:
    subprocess.run([
        sys.executable, "scripts/evaluate_perplexity.py",
        "--model", "outputs/bioqwen/pubmed_pmc_100m/final",
        "--config", "configs/dapt/pubmed_pmc_100m.yaml",
        "--partition", "validation",
        "--tokens", "1000000",
        "--output", "results/perplexity/bioqwen_100m.json"
    ], check=True)


## 9. Qwen-Base vs BioQwen on the Sprint-1 datasets


In [ ]:
RUN_DOWNSTREAM = False
if RUN_DOWNSTREAM:
    subprocess.run([
        sys.executable, "scripts/run_sprint2_suite.py",
        "--config", "configs/suites/sprint2_ablation.yaml"
    ], check=True)


## 10. Aggregate downstream results


In [ ]:
if RUN_DOWNSTREAM:
    subprocess.run([
        sys.executable, "scripts/compare_sprint2.py",
        "--summary", "results/sprint2/summary.json",
        "--output", "results/sprint2/comparison.csv"
    ], check=True)


## 11. Export a Hugging Face adapter directory


In [ ]:
EXPORT_CHECKPOINT = False
if EXPORT_CHECKPOINT:
    subprocess.run([
        sys.executable, "scripts/export_hf.py",
        "--checkpoint", "outputs/bioqwen/pubmed_pmc_100m/final",
        "--output", "exports/BioQwen-4B-100M"
    ], check=True)


## 12. Optional merged Hugging Face export


In [ ]:
MERGE_EXPORT = False
if MERGE_EXPORT:
    subprocess.run([
        sys.executable, "scripts/export_hf.py",
        "--checkpoint", "outputs/bioqwen/pubmed_pmc_100m/final",
        "--output", "exports/BioQwen-4B-100M-merged",
        "--merge"
    ], check=True)
